### 偏差-方差的数学含义

期望预测误差的分解：

139
\mathbb{E}_{\mathcal{D}}\left[(y - \hat{f}_{\mathcal{D}}(x))^2
ight] = \sigma^2 + 	ext{Bias}[\hat{f}(x)]^2 + 	ext{Var}[\hat{f}(x)]
139

- **偏差**：$	ext{Bias}[\hat{f}] = \mathbb{E}[\hat{f}] - f$。模型族对真实函数的系统性偏离——由模型假设容量不足导致
- **方差**：$	ext{Var}[\hat{f}] = \mathbb{E}[(\hat{f} - \mathbb{E}[\hat{f}])^2]$。模型对训练集采样的敏感性——由模型过度灵活导致
- **不可约误差** $\sigma^2$：数据生成过程固有的噪声

偏差和方差通过模型复杂度耦合：降低偏差（加参数）必然增加方差，反之亦然。最优复杂度在两者之间取得平衡。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, KFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, \
    confusion_matrix, roc_curve, auc, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification, make_moons

# === 偏差-方差可视化 ===
np.random.seed(42)  # 固定随机种子，保证结果可复现

# 真实函数
def true_f(x):
    return np.sin(1.5 * np.pi * x)  # 返回结果

# 生成多个训练集，每个拟合一个模型
n_datasets = 20
n_samples = 30
x_test = np.linspace(0, 1, 200)  # 生成等间隔序列

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 创建子图网格
degrees = [1, 4, 15]
titles = ['High Bias (Underfit)', 'Good Fit', 'High Variance (Overfit)']

for ax, deg, title in zip(axes, degrees, titles):
    predictions = []
    for _ in range(n_datasets):
        X_train = np.random.uniform(0, 1, n_samples)  # 均匀分布随机数
        y_train = true_f(X_train) + np.random.normal(0, 0.2, n_samples)
        
        poly = PolynomialFeatures(degree=deg)
        X_poly = poly.fit_transform(X_train.reshape(-1, 1))  # 改变张量形状
        model = LinearRegression().fit(X_poly, y_train)  # 训练模型（拟合数据）
        
        X_test_poly = poly.transform(x_test.reshape(-1, 1))  # 改变张量形状
        pred = model.predict(X_test_poly)  # 用训练好的模型做预测
        predictions.append(pred)
        ax.plot(x_test, pred, alpha=0.15, color='blue', lw=0.5)
    
    mean_pred = np.mean(predictions, axis=0)  # 计算均值
    ax.plot(x_test, true_f(x_test), 'k-', lw=2, label='True f(x)')
    ax.plot(x_test, mean_pred, 'r-', lw=2, label='Mean model')
    
    # 方差区域
    std_pred = np.std(predictions, axis=0)  # 计算标准差
    ax.fill_between(x_test, mean_pred - std_pred, mean_pred + std_pred, alpha=0.2, color='red')
    
    ax.set_title(f'{title}\nDegree={deg}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/bias_variance.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("Key insight:")
print("  Degree 1: High bias — can't capture the sine curve shape")
print("  Degree 4: Good balance — captures the pattern without overfitting to noise")
print("  Degree 15: High variance — model follows every noise point, wildly different from true function")


## 4.2 交叉验证 (Cross-Validation)

单次 train/test split 的评估结果可能依赖于具体的划分方式。**交叉验证**通过多次划分来获得更稳定的评估。

### K-Fold 交叉验证

将数据均分为 $K$ 份，每次用 $K-1$ 份训练，1 份验证，重复 $K$ 次取平均：

$$

\text{CV Score} = \frac{1}{K} \sum_{i=1}^{K} \text{Score}(\text{Model trained on } \mathcal{D}_{-i}, \text{Tested on } \mathcal{D}_i)

$$

典型值：$K=5$ 或 $K=10$。

### 为什么需要交叉验证？

- 稳定评估：减少对单次划分的依赖
- 超参数调优：在验证折上选择最优超参数
- 数据利用：每个样本都有机会参与验证


In [ ]:
# === 交叉验证演示 ===
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge

X_diabetes, y_diabetes = load_diabetes(return_X_y=True)

# 不同 alpha 的 Ridge 回归 CV 比较
alphas = np.logspace(-3, 3, 20)  # 生成对数间隔序列
cv_scores_mean = []
cv_scores_std = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_diabetes, y_diabetes, cv=5,
                             scoring='neg_mean_squared_error')
    cv_scores_mean.append(-scores.mean())  # 沿指定维度求均值
    cv_scores_std.append(scores.std())

# 找最优 alpha
best_idx = np.argmin(cv_scores_mean)
best_alpha = alphas[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))  # 创建子图网格
ax.semilogx(alphas, cv_scores_mean, 'b-', lw=2, label='5-Fold CV MSE')
ax.fill_between(alphas,
                np.array(cv_scores_mean) - np.array(cv_scores_std),  # 创建 NumPy 数组
                np.array(cv_scores_mean) + np.array(cv_scores_std),  # 创建 NumPy 数组
                alpha=0.2, color='blue')
ax.axvline(best_alpha, color='red', linestyle='--',
           label=f'Best alpha={best_alpha:.2f}')
ax.axhline(cv_scores_mean[best_idx], color='green', linestyle='--',
           alpha=0.5, label=f'Best MSE={cv_scores_mean[best_idx]:.1f}')
ax.set_xlabel('Alpha (Regularization Strength)'); ax.set_ylabel('Mean Squared Error')
ax.set_title('Ridge Regression: 5-Fold Cross-Validation for Alpha Selection')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/cross_validation.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print(f"Best alpha: {best_alpha:.3f}, CV MSE: {cv_scores_mean[best_idx]:.1f}")
print(f"\nAlpha=0 (no regularization) MSE: {cv_scores_mean[0]:.1f} — worse!")
print("Cross-validation shows that some regularization IMPROVES generalization.")


## 4.3 正则化 (Regularization)

正则化通过在损失函数中增加**惩罚项**来限制模型复杂度，是防止过拟合的核心技术。

### L1 正则化 (Lasso)

$$

\mathcal{L}_{\text{L1}} = \mathcal{L}_{\text{original}} + \lambda \sum_{i=1}^{d} |w_i|

$$

- 产生**稀疏解**（某些权重精确为 0）→ 自动特征选择
- 适用于特征很多但只有少数重要的场景

### L2 正则化 (Ridge / Weight Decay)

$$

\mathcal{L}_{\text{L2}} = \mathcal{L}_{\text{original}} + \lambda \sum_{i=1}^{d} w_i^2

$$

- 让所有权重趋向于 0 但不为 0
- 对异常值更稳定

### Elastic Net

$$

\mathcal{L}_{\text{EN}} = \mathcal{L}_{\text{original}} + \lambda_1 \sum |w_i| + \lambda_2 \sum w_i^2

$$

### Dropout (深度学习专属)

训练时**随机将一部分神经元输出置 0**（概率 $p$），相当于每次训练一个不同的子网络：

测试时不 dropout，但将权重乘以 $(1-p)$ 以保持期望一致。

**直觉**：Dropout 迫使每个神经元学会"独立有用"的特征，而非依赖其他神经元——相当于对网络进行**集成学习**。


In [ ]:
# === L1 vs L2 对比 ===
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)  # 固定随机种子，保证结果可复现
n_features = 50
n_samples = 100

# 生成数据：只有前 5 个特征有用
X = np.random.randn(n_samples, n_features)  # 标准正态分布随机数
true_w = np.zeros(n_features)  # 创建全零数组
true_w[:5] = [3.0, -2.0, 1.5, -1.0, 0.5]
y = X @ true_w + np.random.randn(n_samples) * 0.5  # 标准正态分布随机数

models = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (L2, alpha=1)': Ridge(alpha=1.0),
    'Lasso (L1, alpha=0.1)': Lasso(alpha=0.1, max_iter=5000),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))  # 创建子图网格
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X, y)  # 训练模型（拟合数据）
    w = model.coef_ if hasattr(model, 'coef_') else model.coef_
    ax.stem(range(n_features), w, basefmt='k-')
    ax.stem(range(5), true_w[:5], linefmt='r-', markerfmt='ro', basefmt='k-')
    ax.set_title(f'{name}')
    ax.set_xlabel('Feature index'); ax.set_ylabel('Weight')
    nnz = np.sum(np.abs(w) > 1e-6)  # 求和
    ax.text(0.95, 0.95, f'Non-zero: {nnz}/{n_features}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10)

plt.suptitle('L1 (Lasso) produces SPARSE solution; L2 (Ridge) shrinks all weights')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/l1_l2_comparison.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 4.4 分类评估指标

准确率 (Accuracy) 是魔鬼——在类别不平衡时完全失效。真正的模型评估需要多种指标。

### 混淆矩阵

| | Predicted Positive | Predicted Negative |
|---|:---:|:---:|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### 核心指标

**精确率 (Precision)**：预测为正的样本中，真正为正的比例

$$

\text{Precision} = \frac{TP}{TP + FP}

$$

**召回率 (Recall / Sensitivity)**：真实正样本中，被正确找出的比例

$$

\text{Recall} = \frac{TP}{TP + FN}

$$

**F1 Score**：精确率和召回率的调和平均

$$

F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}

$$

**ROC 曲线与 AUC**：
- ROC 曲线：以 FPR ($\frac{FP}{FP+TN}$) 为横轴，TPR ($\frac{TP}{TP+FN}$) 为纵轴
- AUC：ROC 曲线下面积——**不依赖于分类阈值**的全局性能度量
- AUC = 1.0（完美）；AUC = 0.5（随机猜测）


In [ ]:
# === 分类指标完整演示 ===
import matplotlib.pyplot as plt
X_cls, y_cls = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, weights=[0.7, 0.3],  # 不平衡 70:30
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.3, random_state=42)  # 将数据划分为训练集和测试集

# 训练模型
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_tr, y_tr)  # 训练模型（拟合数据）
y_pred = model.predict(X_te)  # 用训练好的模型做预测
y_prob = model.predict_proba(X_te)[:, 1]

# 评估（重点：准确率的陷阱）
acc = accuracy_score(y_te, y_pred)
print(f"Accuracy: {acc:.3f}")
print(f"但如果全部预测为多数类 (class 0): {(y_te==0).mean():.3f}  ← 几乎一样!")  # 沿指定维度求均值
print(f"→ 在 70:30 不平衡下，准确率是误导性指标\n")

print(f"Precision: {precision_score(y_te, y_pred):.3f}  (预测为 '正' 的有多少真的为正)")
print(f"Recall:    {recall_score(y_te, y_pred):.3f}  (真实 '正' 的有多少被找出)")
print(f"F1:        {f1_score(y_te, y_pred):.3f}  (综合度量)")
print(f"\n完整报告:")
print(classification_report(y_te, y_pred, target_names=['Negative', 'Positive']))

# 混淆矩阵 + ROC
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 创建子图网格

# 混淆矩阵
cm = confusion_matrix(y_te, y_pred)
axes[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=20,
                     color='white' if cm[i, j] > cm.max()/2 else 'black')  # 沿指定维度取最大值
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred Neg', 'Pred Pos'])
axes[0].set_yticklabels(['True Neg', 'True Pos'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC 曲线
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', lw=1, label='Random (AUC=0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.2, color='blue')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/classification_metrics.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 4.5 学习曲线 (Learning Curves)

学习曲线是诊断**欠拟合 vs 过拟合**的最强工具——用训练集大小做横轴，观察训练/验证误差的变化。

### 诊断规则

| 形态 | 诊断 | 解决 |
|------|------|------|
| 两条曲线都很高且接近 | **欠拟合** | 增加模型复杂度、添加特征 |
| 训练很低、验证很高 | **过拟合** | 正则化、更多数据、简化模型 |
| 两条曲线收敛到低值 | **正好** ✅ | 保持 |

### 过拟合解决方案清单

| 方法 | 原理 | 代价 |
|------|------|------|
| **更多数据** | 样本多→经验分布趋近真实分布 | 成本高 |
| **L1/L2 正则化** | 限制权重范数 | 可能欠拟合 |
| **Dropout** | 训练时随机丢弃神经元 | 训练时间增加 |
| **数据增强** | 对图像/文本做扰动 | 领域特定 |
| **Early Stopping** | 验证误差开始上升时停止 | 简单有效 |
| **简化模型** | 减少层数/神经元 | 可能欠拟合 |
| **Batch Normalization** | 层间归一化，自带正则化效果 | 小 batch 不稳定 |


In [ ]:
# === 学习曲线可视化 ===
import numpy as np
import matplotlib.pyplot as plt
def plot_learning_curve(model, X, y, title, ax):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),  # 生成等间隔序列
        scoring='neg_mean_squared_error'
    )
    train_mean = -train_scores.mean(axis=1)  # 沿指定维度求均值
    train_std = train_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)  # 沿指定维度求均值
    val_std = val_scores.std(axis=1)
    
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training error')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    ax.plot(train_sizes, val_mean, 'o-', color='red', label='Validation error')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='red')
    ax.set_xlabel('Training Set Size'); ax.set_ylabel('MSE')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

X_lc, y_lc = load_diabetes(return_X_y=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 创建子图网格
plot_learning_curve(LinearRegression(), X_lc, y_lc, 'Linear Regression (High Bias)', axes[0])

plot_learning_curve(
    Pipeline([('poly', PolynomialFeatures(degree=10)), ('lr', LinearRegression())]),
    X_lc, y_lc, 'Polynomial deg=10 (Overfit)', axes[1]
)

plot_learning_curve(
    Pipeline([('poly', PolynomialFeatures(degree=2)), ('ridge', Ridge(alpha=1.0))]),
    X_lc, y_lc, 'Polynomial deg=3 + L2 Reg (Good)', axes[2]
)

plt.suptitle('Learning Curves: Diagnosing Bias vs Variance')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/learning_curves.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("\nDiagnosis:")
print("  Left: Both errors high and converge → UNDERFITTING. Need more complexity.")
print("  Middle: Train error low, val error high with gap → OVERFITTING. Need regularization.")
print("  Right: Both errors converge to reasonable level → GOOD FIT.")


## 4.6 Early Stopping

Early Stopping 是最简单也最有效的正则化方法：

1. 每个 epoch 后评估验证集
2. 如果验证误差连续 $N$ 个 epoch 不下降（或上升），停止训练
3. 恢复到验证误差最低点的参数

原理：训练初期，模型学习数据真实模式（验证误差下降）；训练后期，模型开始记忆噪声（验证误差上升）。

```python
# PyTorch Early Stopping 伪代码
best_val_loss = float('inf')
patience_counter = 0
for epoch in range(max_epochs):
    train_loss = train_one_epoch()
    val_loss = validate()
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')  # 保存最佳
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# 恢复最佳参数
model.load_state_dict(torch.load('best_model.pth'))
```


## 本章小结

| 概念 | 一句话总结 |
|------|-----------|
| **Bias-Variance** | 误差 = 偏差² + 方差 + 噪声；偏差和方差不可兼得 |
| **Cross-Validation** | 多次划分取平均，评估更稳定 |
| **L1 正则化** | 产生稀疏权重（特征选择） |
| **L2 正则化** | 收缩所有权重（防过拟合） |
| **Dropout** | 随机丢弃神经元，相当于隐式 ensemble |
| **Precision/Recall** | 不平衡数据下的正确评估指标 |
| **ROC/AUC** | 不依赖阈值的全局性能度量 |
| **Learning Curve** | 诊断欠拟合/过拟合的终极武器 |
| **Early Stopping** | 验证误差不降就停，零成本正则化 |

✅ 掌握这些概念后，你将能够系统地诊断和改进任何机器学习模型。
